# EXP00N — [Short title]

**Question:** [One sentence — the design decision being tested]
**Hypothesis:** [What you expect to find and why]
**Success criterion:** [Specific measurable threshold that would validate the hypothesis]
**Production impact:** [Which function in netweave/src/ would change if validated]
**Author:** Chuck Wiley  **Date:** YYYY-MM-DD

In [ ]:
import sys
sys.path.insert(0, ".")

import mlflow
import mlflow.tracking
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lib.niw_graph import load_graph
from lib.niw_metrics import ndcg_at_k, precision_at_k, build_ground_truth
from lib.niw_mlflow import start_run, log_result, compare_runs

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_ID = "EXP00N"
EXPERIMENT_NAME = "[Short title]"
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
# Always load from shared/ — never modify source data
G = load_graph(
    nodes_path="shared/nodes.parquet",
    edges_path="shared/edges.parquet"
)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Each variant is one value of the design parameter being tested
variants = {
    "variant_A": {"param": "value_A"},
    "variant_B": {"param": "value_B"},
    "variant_C": {"param": "value_C"},
}

In [ ]:
results = []
for variant_name, params in variants.items():
    with mlflow.start_run(run_name=f"{EXPERIMENT_ID}_{variant_name}"):
        mlflow.log_params(params)

        # --- YOUR EXPERIMENT CODE HERE ---
        ranked_contacts = []  # replace with actual ranking
        # ---------------------------------

        relevant_set = build_ground_truth(G, ["tag1", "tag2"])
        score = ndcg_at_k(ranked_contacts, relevant_set)
        mlflow.log_metric("ndcg_at_10", score)
        results.append({"variant": variant_name, "score": score, **params})
        print(f"{variant_name}: {score:.4f}")

results_df = pd.DataFrame(results).sort_values("score", ascending=False)
print(results_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(results_df["variant"], results_df["score"])
ax.set_xlabel("NDCG@10")
ax.set_title(f"{EXPERIMENT_ID}: {EXPERIMENT_NAME}")
plt.tight_layout()
plt.savefig(f"experiments/{EXPERIMENT_ID}/results.png", dpi=150)
plt.show()

## Result and Decision

**Winner:** variant_X (score: N.NN)
**Margin over baseline:** +N.NN
**Decision:** [VALIDATED | INCONCLUSIVE | REJECTED]

**If VALIDATED — production change:**
File: `netweave/src/[filename].py`
Function: `[function_name]`
Change: [one sentence description of what changes]
Notebook citation to add as comment: `# Validated in EXP00N — see netweave-lab/experiments/EXP00N/`